# msarchive — Kaggle runner

Open, permissively-licensed music preservation to the Internet Archive with a
Hugging Face mirror + hourly progress ledger.

- **Author / uploader on every file: Vaibhav Agarwal (archiver)** — nekomeowpurr@proton.me
- Code: https://github.com/thisisforlearn/archive (GPL-3.0-only)
- Source corpus: Free Music Archive full preservation mirror — **please support the original artists**

## Kaggle setup (one-time)

1. Settings → **Internet ON**, accelerator: **CPU** (no GPU needed).
2. **Add-ons → Secrets** with these labels:
   - `IA_S3_ACCESS_KEY`, `IA_S3_SECRET_KEY` (from archive.org/account/s3.php)
   - `HF_TOKEN` (write access to the dataset repo)
3. Save & run all. Every re-run resumes from the HF ledger — nothing is re-extracted or re-uploaded.

Set the Kaggle notebook to **run for the max session (12 h)** — the runner self-stops at **11.5 h**
and does a final flush via SIGTERM-safe signal handling.

In [ ]:
# 1 — clone the archiver code (fresh / resume-agnostic)
import os
SRC = '/kaggle/tmp/msarchive-src'
if not os.path.isdir(SRC):
    !git clone https://github.com/thisisforlearn/archive $SRC
else:
    !cd $SRC && git pull --ff-only || true
print('code ready')


In [ ]:
# 2 — install deps (Kaggle has most; missing ones fast)
!pip install -q -r $SRC/requirements.txt 2>&1 | tail -n 3
import sys; sys.path.insert(0, SRC)
print('deps ok')


In [ ]:
# 3 — Kaggle Secrets → env; bulk storage on scratch (NOT /kaggle/working)
import os, subprocess
secrets_file = '/kaggle/input/__secrets'  # for reference only
try:
    from kaggle_secrets import UserSecretsClient
    usecrets = UserSecretsClient()
    for label in ('IA_S3_ACCESS_KEY', 'IA_ACCESS_KEY', 'IA_S3_SECRET_KEY', 'IA_SECRET_KEY', 'HF_TOKEN'):
        try:
            os.environ.setdefault(label, usecrets.get_secret(label))
            print(f'secret {label}: loaded')
        except Exception as e:
            print(f'secret {label}: MISSING ({e!r})')
except Exception as e:  # local-run fallback
    print('not on Kaggle runner context — using .env only', e)

os.environ.setdefault('MSARCHIVE_HF_REPO_ID', 'thisisforlearn/vaibhav-music-archive')
os.environ.setdefault('MSARCHIVE_CHECKPOINT_EVERY', '3600')
os.environ.setdefault('MSARCHIVE_ARCHIVIST_NAME', 'Vaibhav Agarwal (archiver)')
os.environ.setdefault('MSARCHIVE_ARCHIVIST_EMAIL', 'nekomeowpurr@proton.me')
os.environ.setdefault('MSARCHIVE_ARCHIVIST_GITHUB', 'https://github.com/thisisforlearn/archive')
os.environ.setdefault('MSARCHIVE_DATA_DIR', '/kaggle/tmp/msarchive')
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
print('env ready')


In [ ]:
# 4 — run the full pipeline: restore → scan → [extract ∥ IA-upload] → hourly HF checkpoints
os.chdir(SRC)
!python -m msarchive run --max-hours 11.5


In [ ]:
# 5 — final stats
!python -m msarchive stats | tail -n 4
print('Done — archived by Vaibhav Agarwal (archiver) <nekomeowpurr@proton.me>')
